# 03 MLM Baseline Pre-training
This notebook trains a standard RoBERTa-base model using Masked Language Modeling (MLM) on the ECtHR dataset using the HuggingFace `Trainer`.

In [1]:
import sys
import os
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast, 
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling, 
    Trainer, 
    TrainingArguments,
)

# --- CONFIGURATION ---
CHECKPOINT_DIR = "../checkpoints/mlm"
EPOCHS         = 5
BATCH_SIZE     = 8  # 24GB-32GB VRAM. Scale up if on Spark.
WANDB_PROJECT  = "glocal-nlp"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [2]:
print("Loading and flattening dataset...")
raw_dataset = load_dataset("coastalcph/lex_glue", "ecthr_a")
raw_dataset = raw_dataset.filter(lambda x: len(x["text"]) >= 5)

def flatten_paragraphs(example):
    return {"text": " ".join(example["text"])}

dataset = raw_dataset.map(flatten_paragraphs, remove_columns=["labels"])
print(f"Train size: {len(dataset['train'])}")

Loading and flattening dataset...


Filter:   0%|          | 0/9000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8988 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/992 [00:00<?, ? examples/s]

Train size: 8988


In [3]:
print("Tokenizing...")
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=512, 
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"]
)
tokenized_dataset.set_format("torch")

Tokenizing...


Map:   0%|          | 0/8988 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/992 [00:00<?, ? examples/s]

In [4]:
print("Initializing Trainer...")
model = RobertaForMaskedLM.from_pretrained("roberta-base")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

training_args = TrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    save_steps                  = 1000,
    logging_steps               = 100,
    report_to                   = "wandb",
    run_name                    = "mlm_baseline",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_dataset["train"],
    data_collator = data_collator,
)

print("Starting MLM Training...")
trainer.train()
trainer.save_model(f"{CHECKPOINT_DIR}/mlm_final")
print("MLM Training Complete.")

Initializing Trainer...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/brandomattivi/.netrc.


Starting MLM Training...


wandb: Currently logged in as: brandomattivi (bmattivi03) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/opt/miniconda3/envs/nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x34c9c7810>> (for post_run_cell), with arguments args (<ExecutionResult object at 34c7564d0, execution_count=4 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 34c766d10, raw_cell="print("Initializing Trainer...")
model = RobertaFo.." transformed_cell="print("Initializing Trainer...")
model = RobertaFo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/brandomattivi/Documents/UNI/UNIBZ/MSC/2nd%20semester/nlp/GlocalDoc/notebooks/03_pretrain_mlm.ipynb#W4sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost